# Gold_v1 — Unified Analytical Table
## Medallion Architecture: Silver → Gold

**MSBA 305 — Microsoft Fabric**

---

### Purpose
This notebook joins all three Silver tables into a single, query-ready Gold table.  
Each row in the Gold table represents **one borough × one hour** — the finest grain
at which all three sources can be joined.

### Research Question
> *How does weather affect NYC 311 complaint patterns and traffic volumes?*

### Source Tables
| Silver Table | Grain | Join Key |
|---|---|---|
| `silver_stg_weather` | borough × hour | `borough` + `timestamp` (truncated to hour) |
| `silver_stg_traffic` | borough × hour | `borough` + `hour` |
| `silver_stg_nyc_311` | one row per complaint (SCD Type-2) | `borough` + `date_trunc('hour', created_date)` |

### Join Strategy
Weather is the **left/base table** — it has the most complete hourly coverage across all boroughs.
Both Traffic and 311 are **left-joined**, so we never lose a weather row even if no matching
traffic or complaint data exists for that hour. Availability flags (`traffic_available`,
`complaints_available`) let Power BI filter to fully populated rows when needed.

### Output Table
| Table | Grain | Approx. rows |
|---|---|---|
| `gold_nyc_urban` | borough × hour | ~26,000 (weather-bounded) |

---

### Gold Schema Summary
| # | Column | Source | Description |
|---|---|---|---|
| 1 | `borough` | All | Title case, space-normalised |
| 2 | `hour` | Weather | Timestamp truncated to hour — join anchor |
| 3 | `date` | Derived | Date portion of `hour` |
| 4 | `day_of_week` | Derived | Monday … Sunday |
| 5 | `hour_of_day` | Derived | 0–23 |
| 6 | `is_weekend` | Derived | True for Sat/Sun |
| 7 | `month` | Derived | 1–12 |
| 8–18 | Weather columns | Weather | Temperature, precipitation, wind, etc. |
| 19 | `traffic_volume` | Traffic | Hourly vehicle count |
| 20 | `sensor_count` | Traffic | Number of segments that contributed |
| 21 | `traffic_available` | Derived | False when no traffic row matched |
| 22 | `complaint_count` | 311 | Total complaints that borough+hour |
| 23 | `noise_complaints` | 311 | Noise-related types |
| 24 | `street_complaints` | 311 | Parking, blocked driveway, etc. |
| 25 | `heat_hot_water_complaints` | 311 | Heat / hot water |
| 26 | `other_complaints` | 311 | Remainder |
| 27 | `top_complaint_type` | 311 | Most frequent type that borough+hour |
| 28 | `num_open_complaints` | 311 | Status = Open |
| 29 | `num_closed_complaints` | 311 | Status = Closed |
| 30 | `num_complaints_mobile` | 311 | Channel = MOBILE |
| 31 | `num_complaints_online` | 311 | Channel = ONLINE |
| 32 | `num_complaints_phone` | 311 | Channel = PHONE |
| 33 | `complaints_available` | Derived | False when no 311 row matched |
| 34 | `gold_created_at` | Derived | When this row was written to Gold |


---
## Cell 1 — Imports & Configuration

In [13]:
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# ── Table names ───────────────────────────────────────────────────────────────
SILVER_WEATHER  = "silver_stg_weather"
SILVER_TRAFFIC  = "silver_stg_traffic"
SILVER_311      = "silver_stg_nyc_311"
GOLD_TABLE      = "gold_nyc_urban"

# ── 311 Complaint type buckets ────────────────────────────────────────────────
# These lists define which complaint_type values fall into each named bucket.
# Adjust based on the actual top types in your silver_stg_311 table.
NOISE_TYPES = [
    "Noise - Residential", "Noise - Commercial", "Noise - Street/Sidewalk",
    "Noise - Vehicle", "Noise - Park", "Noise - Helicopter", "Noise"
]

STREET_TYPES = [
    "Blocked Driveway", "Illegal Parking", "Street Condition",
    "Street Light Condition", "Traffic Signal Condition",
    "Roadway Repair", "Highway Condition"
]

HEAT_TYPES = [
    "HEAT/HOT WATER", "Heat", "Hot Water", "Heating"
]

print("✅ Configuration loaded")
print(f"   Weather  : {SILVER_WEATHER}")
print(f"   Traffic  : {SILVER_TRAFFIC}")
print(f"   311      : {SILVER_311}")
print(f"   Gold out : {GOLD_TABLE}")

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 18, Finished, Available, Finished, False)

✅ Configuration loaded
   Weather  : silver_stg_weather
   Traffic  : silver_stg_traffic
   311      : silver_stg_nyc_311
   Gold out : gold_nyc_urban


---
## Cell 2 — Load Silver Tables

Load all three Silver tables and immediately apply **borough normalisation** so the
join key is consistent across all sources before anything else runs.

### Borough normalisation logic
| Source | Raw format | Problem | Fix |
|---|---|---|---|
| Weather | `bronx`, `statenisland` | lowercase, no space | `initcap` + `statenisland → Staten Island` |
| Traffic | `Bronx`, `Brooklyn` | title case, correct | trim only |
| 311 | `Bronx`, `Staten Island` | title case, correct | trim only |

In [14]:
# ── Helper: normalise borough to title case with correct spacing ──────────────
# Applied to all three tables before any join.
# Steps:
#   1. trim()       — remove accidental leading/trailing whitespace
#   2. initcap()    — lowercase → Title Case  (bronx → Bronx)
#   3. regexp_replace — fix 'Statenisland' → 'Staten Island' (initcap collapses the space)
def normalise_borough(col_name):
    return F.regexp_replace(
        F.initcap(F.trim(F.col(col_name))),
        "Statenisland", "Staten Island"
    )

# ── Load Weather ──────────────────────────────────────────────────────────────
# Weather timestamp is already a TimestampType from Silver.
# We truncate to hour here to create the join key.
df_weather = (
    spark.table(SILVER_WEATHER)
    .withColumn("borough", normalise_borough("borough"))
    .withColumn("hour", F.date_trunc("hour", F.col("timestamp")))
)

# ── Load Traffic ──────────────────────────────────────────────────────────────
# Traffic is already aggregated to hourly in Silver — 'hour' column is the join key.
df_traffic = (
    spark.table(SILVER_TRAFFIC)
    .withColumn("borough", normalise_borough("borough"))
)

# ── Load 311 ─────────────────────────────────────────────────────────────────
# 311 is SCD Type-2 — multiple rows may exist per unique_key (one per change_date).
# We keep only the LATEST record per complaint before aggregating,
# so complaint_count reflects the current state of each complaint, not change history.
w_latest = Window.partitionBy("unique_key").orderBy(F.desc("change_date"))

df_311_latest = (
    spark.table(SILVER_311)
    .withColumn("borough", normalise_borough("borough"))
    .withColumn("_rank", F.row_number().over(w_latest))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
)

# Truncate created_date to hour — this is the 311 join key
df_311_latest = df_311_latest.withColumn(
    "complaint_hour", F.date_trunc("hour", F.col("created_date"))
)

# ── Row count summary ─────────────────────────────────────────────────────────
print("Silver tables loaded:")
print(f"   Weather rows  : {df_weather.count():>10,}")
print(f"   Traffic rows  : {df_traffic.count():>10,}")
print(f"   311 rows      : {df_311_latest.count():>10,}  (latest record per complaint)")

print("\nBorough values after normalisation:")
for label, df in [("Weather", df_weather), ("Traffic", df_traffic), ("311", df_311_latest)]:
    vals = sorted([r[0] for r in df.select("borough").distinct().collect()])
    print(f"   {label:<10}: {vals}")

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 19, Finished, Available, Finished, False)

Silver tables loaded:
   Weather rows  :     13,320
   Traffic rows  :        696
   311 rows      :  1,165,545  (latest record per complaint)

Borough values after normalisation:
   Weather   : ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
   Traffic   : ['Bronx', 'Brooklyn', 'Manhattan', 'Queens']
   311       : ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


---
## Cell 3 — Aggregate 311 to Borough × Hour

311 raw data is at complaint grain (one row per complaint). Before joining to Weather,
we aggregate it to **borough × hour** — the same grain as Gold.

Columns produced:
- **`complaint_count`** — total complaints in that borough+hour
- **`noise_complaints`** — count where complaint_type is in the noise bucket
- **`street_complaints`** — count where complaint_type is in the street bucket
- **`heat_hot_water_complaints`** — count where complaint_type is in the heat bucket
- **`other_complaints`** — everything not in a named bucket
- **`top_complaint_type`** — the single most frequent type in that borough+hour
- **`num_open_complaints`** — rows where status = 'Open'
- **`num_closed_complaints`** — rows where status = 'Closed'
- **`num_complaints_mobile`** — channel = 'MOBILE'
- **`num_complaints_online`** — channel = 'ONLINE'
- **`num_complaints_phone`** — channel = 'PHONE'

In [15]:
# ── Step 1: Tag each complaint row with bucket membership ─────────────────────
# Using conditional columns (0 or 1) that sum cleanly in the groupBy.

df_311_tagged = df_311_latest.withColumn(
    "is_noise",
    F.when(F.col("complaint_type").isin(NOISE_TYPES), 1).otherwise(0)
).withColumn(
    "is_street",
    F.when(F.col("complaint_type").isin(STREET_TYPES), 1).otherwise(0)
).withColumn(
    "is_heat",
    F.when(F.col("complaint_type").isin(HEAT_TYPES), 1).otherwise(0)
).withColumn(
    # A complaint counts as 'other' if it didn't match any named bucket
    "is_other",
    F.when(
        ~F.col("complaint_type").isin(NOISE_TYPES + STREET_TYPES + HEAT_TYPES), 1
    ).otherwise(0)
)

# ── Step 2: Find top_complaint_type per borough+hour ─────────────────────────
# Window function ranks complaint types by frequency within each borough+hour.
# We take rank=1 (the most common type) and join it back after the main aggregation.

w_top = Window.partitionBy("borough", "complaint_hour").orderBy(F.desc("_type_count"))

df_top_type = (
    df_311_tagged
    .groupBy("borough", "complaint_hour", "complaint_type")
    .count()
    .withColumnRenamed("count", "_type_count")
    .withColumn("_rank", F.row_number().over(w_top))
    .filter(F.col("_rank") == 1)
    .select(
        F.col("borough").alias("_top_borough"),
        F.col("complaint_hour").alias("_top_hour"),
        F.col("complaint_type").alias("top_complaint_type")
    )
)

# ── Step 3: Main aggregation — all metrics per borough+hour ──────────────────

df_311_agg = (
    df_311_tagged
    .groupBy("borough", "complaint_hour")
    .agg(
        # Total complaints
        F.count("*").alias("complaint_count"),

        # Named complaint buckets
        F.sum("is_noise").alias("noise_complaints"),
        F.sum("is_street").alias("street_complaints"),
        F.sum("is_heat").alias("heat_hot_water_complaints"),
        F.sum("is_other").alias("other_complaints"),

        # Status breakdown
        F.sum(F.when(F.upper(F.col("status")) == "OPEN", 1).otherwise(0))
         .alias("num_open_complaints"),
        F.sum(F.when(F.upper(F.col("status")) == "CLOSED", 1).otherwise(0))
         .alias("num_closed_complaints"),

        # Channel breakdown
        F.sum(F.when(F.upper(F.col("open_data_channel_type")) == "MOBILE", 1).otherwise(0))
         .alias("num_complaints_mobile"),
        F.sum(F.when(F.upper(F.col("open_data_channel_type")) == "ONLINE", 1).otherwise(0))
         .alias("num_complaints_online"),
        F.sum(F.when(F.upper(F.col("open_data_channel_type")) == "PHONE", 1).otherwise(0))
         .alias("num_complaints_phone"),
    )
)

# ── Step 4: Join top_complaint_type back onto the aggregated table ─────────────

df_311_agg = df_311_agg.join(
    df_top_type,
    (df_311_agg["borough"] == df_top_type["_top_borough"]) &
    (df_311_agg["complaint_hour"] == df_top_type["_top_hour"]),
    how="left"
).drop("_top_borough", "_top_hour")

agg_count = df_311_agg.count()
print(f"311 aggregation complete: {agg_count:,} borough×hour combinations")
print("\nSample aggregated 311 rows:")
df_311_agg.orderBy("borough", "complaint_hour").show(5, truncate=False)

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 20, Finished, Available, Finished, False)

311 aggregation complete: 12,923 borough×hour combinations

Sample aggregated 311 rows:
+-------+-------------------+---------------+----------------+-----------------+-------------------------+----------------+-------------------+---------------------+---------------------+---------------------+--------------------+-------------------+
|borough|complaint_hour     |complaint_count|noise_complaints|street_complaints|heat_hot_water_complaints|other_complaints|num_open_complaints|num_closed_complaints|num_complaints_mobile|num_complaints_online|num_complaints_phone|top_complaint_type |
+-------+-------------------+---------------+----------------+-----------------+-------------------------+----------------+-------------------+---------------------+---------------------+---------------------+--------------------+-------------------+
|Bronx  |2026-01-01 00:00:00|74             |38              |15               |12                       |9               |1                  |73              

---
## Cell 4 — Build the Gold Table (Three-Way Join)

### Join order
```
df_weather
  LEFT JOIN df_traffic   ON (borough, hour)
  LEFT JOIN df_311_agg   ON (borough, hour)
```

Weather is the anchor — every borough×hour that has weather data gets a Gold row,
regardless of whether traffic or 311 data exists for that hour.

### Availability flags
After the join, null values in traffic/311 columns indicate no match existed.
We derive boolean flags and fill nulls with 0 for numeric columns.

In [16]:
# ── Prepare Weather columns for Gold ─────────────────────────────────────────
# Select only the columns we want to carry into Gold (drop station metadata).

WEATHER_COLS = [
    "borough", "hour",
    "temperature_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall_mm", "snow_depth",
    "wind_speed_10m", "wind_gusts_10m", "cloud_cover",
    "weather_category", "weather_code",
]

# Only keep weather cols that actually exist in the Silver table
existing_weather_cols = [c for c in WEATHER_COLS if c in df_weather.columns]
df_weather_slim = df_weather.select(existing_weather_cols)

# ── Join 1: Weather LEFT JOIN Traffic ─────────────────────────────────────────
# Prefix traffic columns to avoid ambiguity after the join
df_traffic_prefixed = (
    df_traffic
    .select(
        F.col("borough").alias("t_borough"),
        F.col("hour").alias("t_hour"),
        F.col("traffic_volume"),
        F.col("sensor_count"),
    )
)

df_gold = df_weather_slim.join(
    df_traffic_prefixed,
    (df_weather_slim["borough"] == df_traffic_prefixed["t_borough"]) &
    (df_weather_slim["hour"]    == df_traffic_prefixed["t_hour"]),
    how="left"
).drop("t_borough", "t_hour")

# ── Derive traffic_available flag ─────────────────────────────────────────────
# True = traffic data existed for this borough+hour
# False = left join found no match (traffic_volume will be null → fill with 0)
df_gold = (
    df_gold
    .withColumn("traffic_available", F.col("traffic_volume").isNotNull())
    .withColumn("traffic_volume", F.coalesce(F.col("traffic_volume"), F.lit(0)))
    .withColumn("sensor_count",   F.coalesce(F.col("sensor_count"),   F.lit(0)))
)

# ── Join 2: (Weather+Traffic) LEFT JOIN 311 aggregated ───────────────────────
df_311_prefixed = (
    df_311_agg
    .select(
        F.col("borough").alias("c_borough"),
        F.col("complaint_hour").alias("c_hour"),
        "complaint_count",
        "noise_complaints",
        "street_complaints",
        "heat_hot_water_complaints",
        "other_complaints",
        "top_complaint_type",
        "num_open_complaints",
        "num_closed_complaints",
        "num_complaints_mobile",
        "num_complaints_online",
        "num_complaints_phone",
    )
)

df_gold = df_gold.join(
    df_311_prefixed,
    (df_gold["borough"] == df_311_prefixed["c_borough"]) &
    (df_gold["hour"]    == df_311_prefixed["c_hour"]),
    how="left"
).drop("c_borough", "c_hour")

# ── Derive complaints_available flag + fill nulls ─────────────────────────────
COMPLAINT_COUNT_COLS = [
    "complaint_count", "noise_complaints", "street_complaints",
    "heat_hot_water_complaints", "other_complaints",
    "num_open_complaints", "num_closed_complaints",
    "num_complaints_mobile", "num_complaints_online", "num_complaints_phone",
]

df_gold = df_gold.withColumn(
    "complaints_available", F.col("complaint_count").isNotNull()
)

# Fill all complaint numeric columns with 0 where no match
for col in COMPLAINT_COUNT_COLS:
    df_gold = df_gold.withColumn(col, F.coalesce(F.col(col), F.lit(0)))

print(f"After three-way join: {df_gold.count():,} Gold rows")
print(f"   traffic_available=True  : {df_gold.filter(F.col('traffic_available')).count():,}")
print(f"   complaints_available=True: {df_gold.filter(F.col('complaints_available')).count():,}")

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 21, Finished, Available, Finished, False)

After three-way join: 13,320 Gold rows
   traffic_available=True  : 696
   complaints_available=True: 12,923


---
## Cell 5 — Derive Time Dimension Columns

These columns are derived entirely from `hour` and are useful for slicing in Power BI
without needing a separate date dimension table.

| Column | Logic |
|---|---|
| `date` | `DATE(hour)` |
| `day_of_week` | `date_format(hour, 'EEEE')` → full name |
| `hour_of_day` | `HOUR(hour)` → 0–23 |
| `is_weekend` | day_of_week IN ('Saturday', 'Sunday') |
| `month` | `MONTH(hour)` → 1–12 |

In [17]:
df_gold = (
    df_gold
    .withColumn("date",        F.to_date(F.col("hour")))
    .withColumn("day_of_week", F.date_format(F.col("hour"), "EEEE"))   # Monday, Tuesday…
    .withColumn("hour_of_day", F.hour(F.col("hour")).cast(T.IntegerType()))
    .withColumn("is_weekend",  F.date_format(F.col("hour"), "EEEE").isin("Saturday", "Sunday"))
    .withColumn("month",       F.month(F.col("hour")).cast(T.IntegerType()))
)

print("Time dimension columns derived:")
df_gold.select("borough", "hour", "date", "day_of_week", "hour_of_day", "is_weekend", "month") \
       .show(5, truncate=False)

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 22, Finished, Available, Finished, False)

Time dimension columns derived:
+-------+-------------------+----------+-----------+-----------+----------+-----+
|borough|hour               |date      |day_of_week|hour_of_day|is_weekend|month|
+-------+-------------------+----------+-----------+-----------+----------+-----+
|Bronx  |2026-04-21 04:00:00|2026-04-21|Tuesday    |4          |false     |4    |
|Bronx  |2026-04-21 05:00:00|2026-04-21|Tuesday    |5          |false     |4    |
|Bronx  |2026-04-21 06:00:00|2026-04-21|Tuesday    |6          |false     |4    |
|Bronx  |2026-04-21 07:00:00|2026-04-21|Tuesday    |7          |false     |4    |
|Bronx  |2026-04-21 08:00:00|2026-04-21|Tuesday    |8          |false     |4    |
+-------+-------------------+----------+-----------+-----------+----------+-----+
only showing top 5 rows



---
## Cell 6 — Final Column Order + Metadata

Reorder columns into logical groups matching the schema definition in the header,
and add the `gold_created_at` provenance timestamp.

In [18]:
# ── Add Gold metadata ─────────────────────────────────────────────────────────
df_gold = df_gold.withColumn("gold_created_at", F.current_timestamp())

# ── Define final column order ─────────────────────────────────────────────────
# Groups: join keys → time dims → weather → traffic → 311 → metadata
FINAL_COLS = [
    # Join keys
    "borough", "hour",
    # Time dimensions
    "date", "day_of_week", "hour_of_day", "is_weekend", "month",
    # Weather
    "temperature_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall_mm", "snow_depth",
    "wind_speed_10m", "wind_gusts_10m", "cloud_cover",
    "weather_category", "weather_code",
    # Traffic
    "traffic_volume", "sensor_count", "traffic_available",
    # 311 aggregated
    "complaint_count",
    "noise_complaints", "street_complaints",
    "heat_hot_water_complaints", "other_complaints",
    "top_complaint_type",
    "num_open_complaints", "num_closed_complaints",
    "num_complaints_mobile", "num_complaints_online", "num_complaints_phone",
    "complaints_available",
    # Metadata
    "gold_created_at",
]

# Only select columns that actually exist (guards against optional Silver columns)
existing_final = [c for c in FINAL_COLS if c in df_gold.columns]
missing = [c for c in FINAL_COLS if c not in df_gold.columns]
if missing:
    print(f"⚠️  These expected columns are missing from Gold: {missing}")

df_gold = df_gold.select(existing_final)

print(f"Final Gold schema ({len(existing_final)} columns):")
df_gold.printSchema()

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 23, Finished, Available, Finished, False)

Final Gold schema (34 columns):
root
 |-- borough: string (nullable = true)
 |-- hour: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- month: integer (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall_mm: double (nullable = true)
 |-- snow_depth: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- weather_category: string (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- traffic_volume: long (nullable = false)
 |-- sensor_count: long (nullable = false)
 |-- traffic_available: boolean (nullable = false)
 |-- complaint_count: long (nullable = false)
 |-- n

---
## Cell 7 — Write to Gold (Incremental)

Same incremental pattern as the Silver notebooks:
- Anti-join on `(borough, hour)` against the existing Gold table
- Only write rows that don't already exist
- Log exactly how many rows were written vs skipped

This means the notebook is safe to re-run — it will never double-write.

In [19]:
# ── Check if Gold table already exists ────────────────────────────────────────
try:
    spark.table(GOLD_TABLE).limit(1).collect()
    gold_exists = True
except Exception:
    gold_exists = False

if gold_exists:
    # Anti-join: keep only rows whose (borough, hour) are not already in Gold
    existing_keys = spark.table(GOLD_TABLE).select("borough", "hour")
    df_to_write = df_gold.join(
        existing_keys,
        on=["borough", "hour"],
        how="left_anti"
    )
    already_present = df_gold.count() - df_to_write.count()
    print(f"Gold table exists — incremental write mode")
    print(f"   Rows in this batch  : {df_gold.count():>10,}")
    print(f"   Already in Gold     : {already_present:>10,}  (skipped)")
else:
    df_to_write = df_gold
    print(f"Gold table does not exist — full initial load")

# ── Cache count before write (avoid lazy re-evaluation issue) ─────────────────
rows_to_write = df_to_write.count()

if rows_to_write > 0:
    df_to_write.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(GOLD_TABLE)
    print(f"\n✅ Written {rows_to_write:,} new rows → {GOLD_TABLE}")
else:
    print(f"\n✅ No new rows to write — Gold table is already up to date")

gold_total = spark.table(GOLD_TABLE).count()
print(f"   Total rows in Gold now: {gold_total:,}")

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 24, Finished, Available, Finished, False)

Gold table exists — incremental write mode
   Rows in this batch  :     13,320
   Already in Gold     :          0  (skipped)

✅ Written 13,320 new rows → gold_nyc_urban
   Total rows in Gold now: 13,320


---
## Cell 8 — Verification Queries

Sanity checks to run immediately after writing. Each query targets a specific
concern about the Gold table's correctness.

In [20]:
gold = spark.table(GOLD_TABLE)
gold.createOrReplaceTempView("gold_nyc_urban")

print("═" * 65)
print("  GOLD VERIFICATION QUERIES")
print("═" * 65)

# ── 1. Row count + borough coverage ──────────────────────────────────────────
print("\n── 1. Row count per borough ──")
spark.sql("""
    SELECT borough,
           COUNT(*)                                    AS total_rows,
           MIN(hour)                                   AS earliest_hour,
           MAX(hour)                                   AS latest_hour,
           COUNT(DISTINCT DATE(hour))                  AS unique_days
    FROM gold_nyc_urban
    GROUP BY borough
    ORDER BY borough
""").show(truncate=False)

# ── 2. Duplicate check — should return 0 rows ─────────────────────────────────
print("── 2. Duplicate (borough, hour) check — should be empty ──")
dupes = spark.sql("""
    SELECT borough, hour, COUNT(*) AS n
    FROM gold_nyc_urban
    GROUP BY borough, hour
    HAVING COUNT(*) > 1
""")
dupe_count = dupes.count()
if dupe_count == 0:
    print("   ✅ No duplicate (borough, hour) pairs")
else:
    print(f"   ❌ {dupe_count} duplicate pairs found!")
    dupes.show(10)

# ── 3. Traffic availability ───────────────────────────────────────────────────
print("\n── 3. Traffic availability by borough ──")
spark.sql("""
    SELECT borough,
           SUM(CASE WHEN traffic_available THEN 1 ELSE 0 END)  AS hours_with_traffic,
           SUM(CASE WHEN NOT traffic_available THEN 1 ELSE 0 END) AS hours_no_traffic,
           ROUND(100.0 * SUM(CASE WHEN traffic_available THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_covered
    FROM gold_nyc_urban
    GROUP BY borough
    ORDER BY borough
""").show(truncate=False)

# ── 4. 311 availability ───────────────────────────────────────────────────────
print("── 4. 311 complaints availability by borough ──")
spark.sql("""
    SELECT borough,
           SUM(CASE WHEN complaints_available THEN 1 ELSE 0 END)     AS hours_with_complaints,
           SUM(CASE WHEN NOT complaints_available THEN 1 ELSE 0 END) AS hours_no_complaints,
           SUM(complaint_count)                                       AS total_complaints
    FROM gold_nyc_urban
    GROUP BY borough
    ORDER BY borough
""").show(truncate=False)

# ── 5. Weather category distribution ─────────────────────────────────────────
print("── 5. Weather category distribution ──")
spark.sql("""
    SELECT weather_category, COUNT(*) AS rows
    FROM gold_nyc_urban
    GROUP BY weather_category
    ORDER BY rows DESC
""").show()

# ── 6. Top complaint types overall ───────────────────────────────────────────
print("── 6. Top 10 complaint types (by total count) ──")
spark.sql("""
    SELECT top_complaint_type,
           COUNT(*)                AS hours_as_top,
           SUM(complaint_count)    AS total_complaints
    FROM gold_nyc_urban
    WHERE top_complaint_type IS NOT NULL
    GROUP BY top_complaint_type
    ORDER BY total_complaints DESC
    LIMIT 10
""").show(truncate=False)

# ── 7. Sample Gold rows ───────────────────────────────────────────────────────
print("── 7. Sample Gold rows (5 rows with full data) ──")
spark.sql("""
    SELECT borough, hour, weather_category, temperature_2m,
           traffic_volume, sensor_count,
           complaint_count, top_complaint_type,
           num_open_complaints, num_closed_complaints
    FROM gold_nyc_urban
    WHERE traffic_available = true AND complaints_available = true
    ORDER BY hour
    LIMIT 5
""").show(truncate=False)

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 25, Finished, Available, Finished, False)

═════════════════════════════════════════════════════════════════
  GOLD VERIFICATION QUERIES
═════════════════════════════════════════════════════════════════

── 1. Row count per borough ──
+-------------+----------+-------------------+-------------------+-----------+
|borough      |total_rows|earliest_hour      |latest_hour        |unique_days|
+-------------+----------+-------------------+-------------------+-----------+
|Bronx        |2664      |2026-01-01 00:00:00|2026-04-21 23:00:00|111        |
|Brooklyn     |2664      |2026-01-01 00:00:00|2026-04-21 23:00:00|111        |
|Manhattan    |2664      |2026-01-01 00:00:00|2026-04-21 23:00:00|111        |
|Queens       |2664      |2026-01-01 00:00:00|2026-04-21 23:00:00|111        |
|Staten Island|2664      |2026-01-01 00:00:00|2026-04-21 23:00:00|111        |
+-------------+----------+-------------------+-------------------+-----------+

── 2. Duplicate (borough, hour) check — should be empty ──
   ✅ No duplicate (borough, hour) pai

---
## Cell 9 — Purge Cell (One-Time, Skip in Production)

Run this only when you need to start Gold from scratch.  
After running, skip this cell on all subsequent runs.

In [21]:
# ── ONE-TIME PURGE — skip in production ──────────────────────────────────────
"""
try:
    spark.sql(f"TRUNCATE TABLE {GOLD_TABLE}")
    print(f"✅ Truncated: {GOLD_TABLE}")
except Exception as e:
    print(f"⚠️  {GOLD_TABLE} doesn't exist yet, skipping: {e}")

print("\nPurge complete — re-run from Cell 2.")
"""

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 26, Finished, Available, Finished, False)

'\ntry:\n    spark.sql(f"TRUNCATE TABLE {GOLD_TABLE}")\n    print(f"✅ Truncated: {GOLD_TABLE}")\nexcept Exception as e:\n    print(f"⚠️  {GOLD_TABLE} doesn\'t exist yet, skipping: {e}")\n\nprint("\nPurge complete — re-run from Cell 2.")\n'

---
## Cell 10 — Delete Last 50 Rows (For Incremental Write Testing)

Run this to simulate new data arriving, then re-run Cell 7 to confirm
exactly 50 rows are written back.

In [22]:
# ── Delete last 50 rows for incremental write testing ────────────────────────
"""
from delta.tables import DeltaTable

cutoff = (
    spark.table(GOLD_TABLE)
    .orderBy(F.col("hour").desc())
    .limit(50)
    .select("borough", "hour")
)

print("Rows to be deleted:")
cutoff.show(10, truncate=False)

dt = DeltaTable.forName(spark, GOLD_TABLE)
dt.delete(
    F.struct("borough", "hour").isin(
        [F.struct(F.lit(r["borough"]), F.lit(r["hour"])) for r in cutoff.collect()]
    )
)

after_count = spark.table(GOLD_TABLE).count()
print(f"✅ Deleted 50 rows — Gold now has {after_count:,} rows")
print(f"   Re-run Cell 7 and confirm it writes exactly 50 rows back.")
"""

StatementMeta(, 23371df4-59d4-47d9-9f33-7a95f5933950, 27, Finished, Available, Finished, False)

'\nfrom delta.tables import DeltaTable\n\ncutoff = (\n    spark.table(GOLD_TABLE)\n    .orderBy(F.col("hour").desc())\n    .limit(50)\n    .select("borough", "hour")\n)\n\nprint("Rows to be deleted:")\ncutoff.show(10, truncate=False)\n\ndt = DeltaTable.forName(spark, GOLD_TABLE)\ndt.delete(\n    F.struct("borough", "hour").isin(\n        [F.struct(F.lit(r["borough"]), F.lit(r["hour"])) for r in cutoff.collect()]\n    )\n)\n\nafter_count = spark.table(GOLD_TABLE).count()\nprint(f"✅ Deleted 50 rows — Gold now has {after_count:,} rows")\nprint(f"   Re-run Cell 7 and confirm it writes exactly 50 rows back.")\n'